In [3]:
import pandas as pd
import re

# 1. Bring in the specific cleaning functions from your preprocessing pipeline
def clean_transmission_type(x):
    if pd.isna(x): return "Unknown"
    val = str(x).split('(')[0].strip()
    if val in ['Automatic', 'Manual']: return val
    return 'Other'

def extract_cylinders(x):
    if pd.isna(x) or str(x).strip() == '': return "Unknown"
    c_val = "Other"
    cyl = re.search(r'([VIW])[- ]?(\d+)', x, re.IGNORECASE)
    flat = re.search(r'Flat[- ]?(\d+)', x, re.IGNORECASE)
    inline = re.search(r'Inline[- ]?(\d+)', x, re.IGNORECASE)
    l_typo = re.search(r'(l)(\d+)', x)
    
    if cyl: c_val = cyl.group(1).upper() + cyl.group(2)
    elif inline: c_val = 'I' + inline.group(1)
    elif flat: c_val = 'H' + flat.group(1) 
    elif l_typo: c_val = 'I' + l_typo.group(2)
    elif 'Rotary' in x: c_val = 'Rotary'
    elif 'Electric' in x or 'Motor' in x: c_val = 'Electric'
    return c_val
    
def clean_model(x):
    if pd.isna(x):
        return x
    return x.replace('\nSave', '').strip()

# 2. Load your raw historical data (update the filename if you use v2 or v3)
df = pd.read_csv('../data/cars_and_bids_full_history_v3.csv')

# 3. Apply the functions to create the clean columns the frontend expects
df['Transmission_Type'] = df['Transmission'].apply(clean_transmission_type)
df['Engine_Cylinders'] = df['Engine'].apply(extract_cylinders)
df['Model'] = df['Model'].apply(clean_model)

# 4. Keep the required columns
cols_to_keep = ['Make', 'Model', 'Drivetrain', 'Body Style', 'Transmission_Type', 'Engine_Cylinders']

# 5. Drop missing values and duplicates to make the file tiny
dropdown_df = df[cols_to_keep].dropna().drop_duplicates()

# 6. Save to the new file for Streamlit
dropdown_df.to_csv('../data/frontend_data/dropdown_options.csv', index=False)